# 02 — Claim Detector Evaluation

**Why this notebook exists.** `Rahilgh/greentruth-claim-detector` is published and the
application loads it, but **its accuracy has never been measured**. Notebook 01 was saved
without outputs, so no precision / recall / F1 for these weights exists in the repository
or on the Hub. Until this notebook is run, GreenTruth cannot honestly display a detection
metric anywhere — and it does not.

The model card currently says "the paper reports macro-F1 around 85% for comparable
models". That is the **source paper's** number for a *different* model. It is not a
measurement of these weights and must never be quoted as ours.

## What this measures

| | |
|---|---|
| **Model** | `Rahilgh/greentruth-claim-detector` (downloaded from the Hub) |
| **Data** | `climatebert/environmental_claims` test split (downloaded from the Hub) |
| **Baseline** | GreenTruth's own rule-based detector, on the identical split |
| **Metrics** | accuracy, precision, recall, F1 (per class + macro), confusion matrix, threshold sweep, calibration |

The rule-based comparison is the point: the application falls back to those rules whenever
`transformers` is unavailable, so the honest question is not "is the transformer good?" but
**"how much does the transformer actually buy over the fallback that ships by default?"**

## Requirements

Nothing to upload. Both the dataset and the model download themselves. Colab CPU is
enough (this is inference only, ~2,600 sentences).

> **Leakage note.** These weights were fine-tuned on this dataset's *train* split in
> notebook 01. Evaluating on its *test* split is the standard protocol and is what the
> dataset is designed for — but it is in-domain. Section 7 therefore also reports
> performance on sentences drawn from the GreenTruth demo corpus, which is out-of-domain
> and much smaller, and labels it as indicative rather than a benchmark.

| Notebook card | |
|---|---|
| **Type** | Evaluate a model |
| **Purpose** | Measure the published detector on the held-out test split and compare it with the rule-based fallback. |
| **Inputs** | The published model and the `climatebert/environmental_claims` test split (both downloaded). |
| **Outputs** | `02_detector_metrics.json` → `notebooks/executed/results/`. |
| **Where it runs** | Google Colab (CPU is enough). The rule-based arm could not import the package in Colab; it is reproduced by `scripts/eval_rule_detector.py` on the same split. |
| **Execution record** | Executed in Colab — record in `notebooks/executed/02_claim_detector_evaluation.ipynb`; numbers in `evaluation/experiment_registry.json` and `RESULTS.md`. |


## 1. Install and import

In [ ]:
# Colab has torch already; datasets/transformers may need installing.
try:
    import transformers, datasets            # noqa: F401
except ImportError:
    %pip install -q "transformers>=4.40" "datasets>=2.18" scikit-learn

import json, numpy as np, pandas as pd, torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, classification_report,
                             roc_auc_score, precision_recall_curve, brier_score_loss)
import matplotlib.pyplot as plt

MODEL_ID   = "Rahilgh/greentruth-claim-detector"
DATASET_ID = "climatebert/environmental_claims"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| transformers:", transformers.__version__)

## 2. Data — downloaded, not uploaded

`climatebert/environmental_claims` (Stammbach et al., ACL 2023): 2,647 expert-annotated
sentences from corporate annual reports, sustainability reports and earnings calls.
Label 1 = environmental claim. Licence CC BY-NC-SA 4.0.
<https://huggingface.co/datasets/climatebert/environmental_claims>

In [ ]:
ds = load_dataset(DATASET_ID)
print(ds)
test = ds["test"]
texts  = test["text"]
y_true = np.array(test["label"])
print(f"\ntest sentences: {len(texts)}")
print(f"class balance : {np.bincount(y_true)}  "
      f"(positive rate {y_true.mean():.3f})")
print("\nexamples:")
for i in range(3):
    print(f"  [{y_true[i]}] {texts[i][:110]}")

## 3. Model — downloaded from the Hub

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID).to(DEVICE).eval()
print("id2label:", model.config.id2label)

POS_ID = next((int(i) for i, l in model.config.id2label.items()
               if "claim" in str(l).lower() and "not" not in str(l).lower()), 1)
print("positive class id:", POS_ID, "->", model.config.id2label[POS_ID])

@torch.no_grad()
def predict_proba(sentences, batch_size=32):
    out = []
    for i in range(0, len(sentences), batch_size):
        b = tok(sentences[i:i + batch_size], truncation=True, max_length=256,
                padding=True, return_tensors="pt").to(DEVICE)
        out.append(torch.softmax(model(**b).logits, dim=-1)[:, POS_ID].cpu().numpy())
    return np.concatenate(out)

probs = predict_proba(list(texts))
print(f"\nscored {len(probs)} sentences | mean p(claim) = {probs.mean():.3f}")

## 4. Headline metrics at the default 0.5 threshold

In [ ]:
def report(y_true, y_pred, name):
    acc = accuracy_score(y_true, y_pred)
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average=None,
                                                 labels=[0, 1], zero_division=0)
    pm, rm, fm, _ = precision_recall_fscore_support(y_true, y_pred, average="macro",
                                                    zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    return dict(model=name, accuracy=float(acc),
                precision_not_claim=float(p[0]), recall_not_claim=float(r[0]),
                f1_not_claim=float(f[0]),
                precision_claim=float(p[1]), recall_claim=float(r[1]),
                f1_claim=float(f[1]),
                macro_precision=float(pm), macro_recall=float(rm), macro_f1=float(fm),
                confusion_matrix=cm.tolist(), n=int(len(y_true)))

y_pred = (probs >= 0.5).astype(int)
bert = report(y_true, y_pred, "climatebert (threshold 0.5)")

print("=" * 66)
print("MEASURED ON THE TEST SPLIT — these are GreenTruth's own numbers")
print("=" * 66)
print(classification_report(y_true, y_pred, target_names=["not_claim", "claim"],
                            digits=4, zero_division=0))
print("confusion matrix [rows=true, cols=pred]:")
print(pd.DataFrame(bert["confusion_matrix"],
                   index=["true not_claim", "true claim"],
                   columns=["pred not_claim", "pred claim"]))
print(f"\nROC AUC   : {roc_auc_score(y_true, probs):.4f}")
print(f"Brier     : {brier_score_loss(y_true, probs):.4f}")

## 5. The comparison that matters — transformer vs the shipped fallback

`greentruth/detectors.py` uses `RuleBasedDetector` whenever `transformers` is not
installed, which is the default deployment. So the number that actually matters for the
product is the **gap between the two**, measured on the same sentences.

The rule detector requires (a) a known environmental metric AND (b) a number or an
achievement/commitment verb. It is tuned for precision: a false positive becomes a verdict
about something nobody claimed.

In [ ]:
import sys, os

# Import the REAL rule detector from the repository rather than re-implementing
# it here, so this measures the code that actually ships. No fabricated repo URL:
# make greentruth/ reachable by one of the routes printed below.
for cand in (".", "..", "/content", "/content/greentruth", "/content/drive/MyDrive/greentruth"):
    if os.path.isdir(os.path.join(cand, "greentruth")) and cand not in sys.path:
        sys.path.insert(0, cand)

try:
    from greentruth.detectors import RuleBasedDetector
    HAVE_REPO = True
    print("rule-based detector imported from the repository")
except Exception as e:
    HAVE_REPO = False
    print(f"Could not import greentruth.detectors ({type(e).__name__}).")
    print("The transformer numbers above are unaffected; only the baseline arm is skipped.")
    print()
    print("To include it, make the repo importable by ONE of:")
    print("  * Files panel -> upload the greentruth/ folder next to this notebook")
    print("  * from google.colab import drive; drive.mount('/content/drive')")
    print("    with the project at /content/drive/MyDrive/greentruth")
    print("  * run this notebook from inside a local checkout")

if HAVE_REPO:
    rb = RuleBasedDetector()
    y_rule = np.array([int(rb.score(t)[0]) for t in texts])
    rule = report(y_true, y_rule, "rule_based (shipped fallback)")
    comp = pd.DataFrame([bert, rule])[
        ["model", "accuracy", "macro_f1", "precision_claim", "recall_claim", "f1_claim"]]
    print(comp.round(4).to_string(index=False))
    d_f1 = bert["macro_f1"] - rule["macro_f1"]
    print(f"\nmacro-F1 difference (transformer - rules): {d_f1:+.4f}")
    print("Interpretation: this is what installing transformers actually buys on this")
    print("dataset. If it is small, the zero-dependency default is scientifically fine.")
else:
    rule = None

## 6. Threshold sweep and calibration

`detectors.py` currently uses the untuned default threshold of 0.5. If another threshold
is materially better on this split, that is a concrete change to make — but note it would
be tuned on **test**, so treat it as diagnostic unless re-derived on a validation split.

In [ ]:
rows = []
for t in np.arange(0.05, 0.96, 0.05):
    yp = (probs >= t).astype(int)
    pm, rm, fm, _ = precision_recall_fscore_support(y_true, yp, average="macro",
                                                    zero_division=0)
    rows.append(dict(threshold=round(float(t), 2), macro_f1=fm,
                     macro_precision=pm, macro_recall=rm,
                     accuracy=accuracy_score(y_true, yp)))
sweep = pd.DataFrame(rows)
best = sweep.loc[sweep.macro_f1.idxmax()]
print(sweep.round(4).to_string(index=False))
print(f"\nbest macro-F1 {best.macro_f1:.4f} at threshold {best.threshold}")
print(f"default 0.5 gives {bert['macro_f1']:.4f}  "
      f"(difference {best.macro_f1 - bert['macro_f1']:+.4f})")

fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
ax[0].plot(sweep.threshold, sweep.macro_f1, marker="o", label="macro F1")
ax[0].plot(sweep.threshold, sweep.macro_precision, marker="s", alpha=.6, label="macro P")
ax[0].plot(sweep.threshold, sweep.macro_recall, marker="^", alpha=.6, label="macro R")
ax[0].axvline(0.5, color="r", ls="--", lw=1, label="shipped default")
ax[0].set_xlabel("threshold"); ax[0].set_title("Threshold sweep"); ax[0].legend(fontsize=8)

pr, rc, _ = precision_recall_curve(y_true, probs)
ax[1].plot(rc, pr); ax[1].set_xlabel("recall"); ax[1].set_ylabel("precision")
ax[1].set_title("Precision-recall")

# reliability diagram: are the scores calibrated probabilities?
bins = np.linspace(0, 1, 11)
idx = np.digitize(probs, bins) - 1
xs, ys, ns = [], [], []
for b in range(10):
    m = idx == b
    if m.sum() >= 5:
        xs.append(probs[m].mean()); ys.append(y_true[m].mean()); ns.append(int(m.sum()))
ax[2].plot([0, 1], [0, 1], "k--", lw=1, label="perfect")
ax[2].plot(xs, ys, marker="o", label="observed")
ax[2].set_xlabel("mean predicted p(claim)"); ax[2].set_ylabel("observed frequency")
ax[2].set_title("Reliability"); ax[2].legend(fontsize=8)
plt.tight_layout(); plt.show()
print("bin counts:", ns)

## 7. Out-of-domain spot check

The test split above is in-domain: same corpus, same annotation protocol, and these
weights were fine-tuned on its train split. A handful of GreenTruth-style sentences gives
a rough sense of transfer.

**This is indicative, not a benchmark** — it is a tiny hand-written set, so treat it as an
error-analysis aid and do not quote a number from it.

In [ ]:
# Hand-written probes. Labels are the author's judgement of "is this a checkable
# environmental claim", matching the dataset's definition as closely as possible.
probe = [
    ("We reduced routine gas flaring by 40% from 2019 levels.", 1),
    ("We will eliminate routine flaring by 2030.", 1),
    ("Flaring decreased by 40% between 2019 and 2024.", 1),
    ("Our methane intensity fell to 0.2% in 2023.", 1),
    ("Reducing flaring is central to our roadmap.", 0),
    ("We care deeply about the environment.", 0),
    ("The board met four times during the reporting period.", 0),
    ("Revenue grew 12% year on year.", 0),
    ("Our teams are passionate about sustainability.", 0),
    ("We installed 40 MW of solar capacity in 2023.", 1),
]
pt = [p[0] for p in probe]; py = np.array([p[1] for p in probe])
pp = predict_proba(pt)
out = pd.DataFrame(dict(sentence=[s[:62] for s in pt], label=py,
                        p_claim=pp.round(3), pred=(pp >= 0.5).astype(int)))
out["correct"] = out.label == out.pred
print(out.to_string(index=False))
print(f"\n{out.correct.sum()}/{len(out)} correct on this small probe set "
      f"(indicative only, n={len(out)})")

## 8. Save the measured metrics

Written to `02_detector_metrics.json`. Download it into the repository so the
application and the model card can quote **measured** numbers instead of the paper's.

In [ ]:
results = dict(
    measured_on="climatebert/environmental_claims test split",
    model_id=MODEL_ID,
    n_test=int(len(y_true)),
    positive_rate=float(y_true.mean()),
    climatebert=bert,
    rule_based=(rule if HAVE_REPO else None),
    roc_auc=float(roc_auc_score(y_true, probs)),
    brier=float(brier_score_loss(y_true, probs)),
    threshold_sweep=sweep.round(5).to_dict("records"),
    best_threshold=float(best.threshold),
    best_threshold_macro_f1=float(best.macro_f1),
    shipped_threshold=0.5,
    caveats=[
        "In-domain: these weights were fine-tuned on this dataset's train split.",
        "The best threshold above was selected on the TEST split and is diagnostic "
        "only; re-derive it on validation before shipping it.",
        "Detecting a claim is not judging it. These metrics say nothing about "
        "verification accuracy, which is what notebooks 04-06 measure.",
    ],
)
with open("02_detector_metrics.json", "w") as f:
    json.dump(results, f, indent=2)

print("=" * 66)
print("COPY THESE INTO THE MODEL CARD (they are measured, not quoted)")
print("=" * 66)
print(f"  accuracy        {bert['accuracy']:.4f}")
print(f"  macro precision {bert['macro_precision']:.4f}")
print(f"  macro recall    {bert['macro_recall']:.4f}")
print(f"  macro F1        {bert['macro_f1']:.4f}")
print(f"  claim-class F1  {bert['f1_claim']:.4f}")
print(f"  ROC AUC         {results['roc_auc']:.4f}")
if HAVE_REPO:
    print(f"  rule baseline macro F1  {rule['macro_f1']:.4f}")
print("\nsaved -> 02_detector_metrics.json")
print("\nNext: download it to the repo root, then update")
print("notebooks/model_card_greentruth_claim_detector.md with these figures and")
print("delete the sentence quoting the source paper's ~85%.")

## 9. Optionally push the corrected model card

Only the README is updated — **the weights are not touched**. Create a Colab secret named
`HF_TOKEN` (🔑 in the left sidebar) with a **write** token from
<https://huggingface.co/settings/tokens>. Never paste a token into a cell.

In [ ]:
PUSH_CARD = False       # <-- set True to update the model card on the Hub

if PUSH_CARD:
    from huggingface_hub import HfApi
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        import os
        HF_TOKEN = os.environ.get("HF_TOKEN")
    assert HF_TOKEN, "no HF_TOKEN found — create the Colab secret first"

    card = f'''---
license: cc-by-nc-sa-4.0
language: en
base_model: climatebert/distilroberta-base-climate-f
datasets: [climatebert/environmental_claims]
tags: [text-classification, climate, esg, greenwashing]
---

# GreenTruth Claim Detector

Binary classifier flagging whether a sentence is an environmental claim. Text stage of
GreenTruth, which checks environmental claims against Earth-observation evidence.

## Intended use
Find checkable environmental claims in corporate disclosures, before evidence-grounded
verification. **Detecting a claim is not judging it.**

## Training
Fine-tuned `climatebert/distilroberta-base-climate-f` on `climatebert/environmental_claims`
(Stammbach et al., ACL 2023): 3 epochs, lr 2e-5, batch 16, max length 256.

## Evaluation — measured, not quoted

Measured on the held-out **test** split (n = {bert["n"]}), notebook
`02_claim_detector_evaluation.ipynb`, threshold 0.5:

| metric | value |
|---|---|
| accuracy | {bert["accuracy"]:.4f} |
| macro precision | {bert["macro_precision"]:.4f} |
| macro recall | {bert["macro_recall"]:.4f} |
| macro F1 | {bert["macro_f1"]:.4f} |
| claim-class F1 | {bert["f1_claim"]:.4f} |
| ROC AUC | {results["roc_auc"]:.4f} |

Confusion matrix (rows = true, cols = predicted): `{bert["confusion_matrix"]}`

These are this model's own numbers. Earlier versions of this card quoted ~85% macro-F1
from the source paper for *comparable* models; that was never a measurement of these
weights and has been removed.

## Limitations
English, listed-company disclosure text; domain shift is likely. In-domain evaluation
(fine-tuned on this dataset's train split). Detects a claim, not its truth. Inherits
CC BY-NC-SA 4.0 (non-commercial). Do not use it to label a company deceptive.

## Failure cases
Claim-like framing with no commitment ("reducing flaring is central to our roadmap") and
quantified non-environmental statements ("revenue grew 12%") are the observed confusions.

## Dataset
https://huggingface.co/datasets/climatebert/environmental_claims
'''
    open("README.md", "w").write(card)
    HfApi(token=HF_TOKEN).upload_file(
        path_or_fileobj="README.md", path_in_repo="README.md",
        repo_id=MODEL_ID, repo_type="model")
    print(f"model card updated -> https://huggingface.co/{MODEL_ID}")
else:
    print("PUSH_CARD is False — nothing uploaded. The weights are never modified by")
    print("this notebook; only README.md would be replaced.")